# Day 043 — LightGBM Deep Dive: Leaf-Wise Growth, GOSS, EFB & Native Categoricals

> **Approach:** Architectural Deep Dive & Mathematical Principles → Hands-on Code & Performance Benchmarking
> **No prior knowledge required** beyond XGBoost basics.

---

## 📌 Key Topics Covered
1. **LightGBM vs. XGBoost**: Architectural innovations behind 10x speedups & lower memory footprint
2. **Leaf-Wise (Best-First) Tree Growth**: Why splitting nodes with maximum loss reduction optimizes convergence (and controlling `num_leaves` to prevent overfitting)
3. **Histogram-Based Split Finding**: Continuous feature binning & $O(\text{bins})$ histogram subtraction trick
4. **Gradient-Based One-Side Sampling (GOSS)**: Downsampling low-gradient data while preserving accurate gradient statistics
5. **Exclusive Feature Bundling (EFB)**: Graph-coloring-based bundling of mutually exclusive sparse features
6. **Native Categorical Feature Handling**: $O(K \log K)$ optimal partition without One-Hot Encoding
7. **LightGBM Python APIs**: `LGBMClassifier` / `LGBMRegressor` vs. Native `lgb.Dataset` & `lgb.train()`
8. **Hyperparameter Tuning & Benchmarking**: Practical tuning rules for `num_leaves`, `min_child_samples`, `feature_fraction`


## Part 1 — Setup & Environment Imports
We import `lightgbm`, `xgboost`, `scikit-learn`, `numpy`, `pandas`, and `matplotlib`.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
import xgboost as xgb
import time
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error, roc_auc_score

np.random.seed(42)
print("Setup completed successfully!")

Setup completed successfully!


## Part 2 — LightGBM Architectural Breakthroughs

### 1. Leaf-Wise (Best-First) vs. Level-Wise (Depth-Wise) Growth
- **Level-Wise (XGBoost Default)**: Grows the tree level by level, splitting all nodes at depth $d$ before moving to depth $d+1$. Balances tree structure but spends computation on nodes with low loss reduction.
- **Leaf-Wise (LightGBM)**: Selects the single node with the **maximum delta loss reduction (Gain)** across the entire tree and splits only that node.

$$\text{Node}_{\text{target}} = \arg\max_{\text{leaf } j} \text{Gain}(j)$$

> ⚠️ **Overfitting Control**: Leaf-wise growth can create deep asymmetrical trees. To control complexity, tune **`num_leaves`** (Rule of thumb: `num_leaves` $\le 2^{\text{max\_depth}}$).

---

In [2]:
# Create synthetic dataset for leaf-wise demonstration
X_cls, y_cls = make_classification(n_samples=5000, n_features=20, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X_cls, y_cls, test_size=0.2, random_state=42)

# Model with controlled num_leaves vs deep max_depth
model_leafwise = lgb.LGBMClassifier(
    num_leaves=31,
    max_depth=6,
    learning_rate=0.05,
    n_estimators=100,
    random_state=42,
    verbosity=-1
)
model_leafwise.fit(X_train, y_train)
preds = model_leafwise.predict(X_test)
print(f"Leaf-Wise Model Test Accuracy: {accuracy_score(y_test, preds):.4f}")

Leaf-Wise Model Test Accuracy: 0.9170


## Part 3 — Histogram-Based Split Finding & Subtraction Trick

1. **Feature Binning**: Continuous feature values are discretized into $K$ discrete bins (typically $K=255$).
2. **Computational Complexity**: Reduces search per split from $O(\text{data} \times \text{features})$ to $O(\text{bins} \times \text{features})$.
3. **Histogram Subtraction**: Once the parent histogram and one child histogram are constructed, the other child's histogram is computed instantly via vector subtraction:

$$\text{Hist}_{\text{Right}} = \text{Hist}_{\text{Parent}} - \text{Hist}_{\text{Left}}$$

## Part 4 — GOSS (Gradient-Based One-Side Sampling) & EFB (Exclusive Feature Bundling)

### 1. GOSS (Gradient-Based One-Side Sampling)
- Instances with **large gradients** ($g_i$) contribute more to information gain.
- GOSS retains all top $a\%$ instances with largest gradients and randomly samples $b\%$ of small-gradient instances.
- To maintain data distribution balance, small-gradient samples are multiplied by $\frac{1-a}{b}$ when computing gain.

### 2. EFB (Exclusive Feature Bundling)
- High-dimensional sparse features (e.g. One-Hot vectors) rarely take non-zero values simultaneously.
- EFB binds mutually exclusive features into a single bundled feature using graph coloring heuristics, dramatically reducing feature count.

In [3]:
# Benchmark standard LightGBM (gbdt) vs GOSS sampling
X_goss, y_goss = make_classification(n_samples=20000, n_features=30, random_state=42)

# Standard GBDT
t0 = time.time()
model_gbdt = lgb.LGBMClassifier(boosting_type='gbdt', n_estimators=100, random_state=42, verbosity=-1)
model_gbdt.fit(X_goss, y_goss)
t_gbdt = time.time() - t0

# GOSS Boosting
t0 = time.time()
model_goss = lgb.LGBMClassifier(boosting_type='goss', top_rate=0.2, other_rate=0.1, n_estimators=100, random_state=42, verbosity=-1)
model_goss.fit(X_goss, y_goss)
t_goss = time.time() - t0

print(f"Standard GBDT Training Time: {t_gbdt:.4f} sec")
print(f"GOSS Boosting Training Time:  {t_goss:.4f} sec (Speedup: {t_gbdt/t_goss:.2f}x)")

Standard GBDT Training Time: 0.5405 sec
GOSS Boosting Training Time:  0.5653 sec (Speedup: 0.96x)


## Part 5 — Native Categorical Feature Handling

Instead of expanding categorical variables into sparse $O(K)$ One-Hot columns:
1. LightGBM sorts categories by their gradient/hessian ratio $\frac{\sum g_i}{\sum h_i + \lambda}$.
2. It finds the optimal binary split subset in $O(K \log K)$ time complexity.
3. Superior split accuracy with zero memory explosion.

In [9]:
# Create synthetic dataset with categorical feature
df_cat = pd.DataFrame({
    'cat_feature': pd.Series(np.random.choice(['NYC', 'LON', 'PAR', 'TOK', 'BER'], size=1000), dtype='category'),
    'num_feature': np.random.randn(1000),
    'target': np.random.randint(0, 2, size=1000)
})

X_c = df_cat[['cat_feature', 'num_feature']]
y_c = df_cat['target']

model_cat = lgb.LGBMClassifier(n_estimators=50, random_state=42, verbosity=-1)
model_cat.fit(X_c, y_c)
print("Native categorical model trained successfully!")

Native categorical model trained successfully!


## Part 6 — LightGBM Native `lgb.Dataset` & `lgb.train` API
Similar to XGBoost's `DMatrix`, LightGBM provides `lgb.Dataset` for direct memory management, custom evaluation metrics, and validation early stopping.

In [10]:
# Prepare Native lgb.Dataset
X_tr, X_val, y_tr, y_val = train_test_split(X_cls, y_cls, test_size=0.2, random_state=42)
dtrain_lgb = lgb.Dataset(X_tr, label=y_tr)
dval_lgb = lgb.Dataset(X_val, label=y_val, reference=dtrain_lgb)

params = {
    'objective': 'binary',
    'metric': ['binary_logloss', 'auc'],
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'verbose': -1,
    'seed': 42
}

# Train with early stopping
evals_result = {}
bst_lgb = lgb.train(
    params,
    dtrain_lgb,
    num_boost_round=100,
    valid_sets=[dtrain_lgb, dval_lgb],
    valid_names=['train', 'val'],
    callbacks=[lgb.early_stopping(stopping_rounds=10, verbose=False), lgb.log_evaluation(period=20)]
)

val_preds = bst_lgb.predict(X_val)
print(f"\nNative LGBM Validation AUC: {roc_auc_score(y_val, val_preds):.4f}")


Native LGBM Validation AUC: 0.9739


In [11]:
# Benchmark XGBoost vs LightGBM on speed and accuracy
X_bench, y_bench = make_classification(n_samples=50000, n_features=30, random_state=42)
X_b_tr, X_b_te, y_b_tr, y_b_te = train_test_split(X_bench, y_bench, test_size=0.2, random_state=42)

# 1. XGBoost
t0 = time.time()
model_xgb_bench = xgb.XGBClassifier(n_estimators=100, max_depth=6, tree_method='hist', random_state=42)
model_xgb_bench.fit(X_b_tr, y_b_tr)
t_xgb = time.time() - t0
acc_xgb = accuracy_score(y_b_te, model_xgb_bench.predict(X_b_te))

# 2. LightGBM
t0 = time.time()
model_lgb_bench = lgb.LGBMClassifier(n_estimators=100, num_leaves=63, random_state=42, verbosity=-1)
model_lgb_bench.fit(X_b_tr, y_b_tr)
t_lgb = time.time() - t0
acc_lgb = accuracy_score(y_b_te, model_lgb_bench.predict(X_b_te))

print("--- XGBoost (hist) vs LightGBM Benchmark (50,000 samples) ---")
print(f"XGBoost  (hist): Time = {t_xgb:.4f}s | Accuracy = {acc_xgb:.4f}")
print(f"LightGBM (goss): Time = {t_lgb:.4f}s | Accuracy = {acc_lgb:.4f}")

--- XGBoost (hist) vs LightGBM Benchmark (50,000 samples) ---
XGBoost  (hist): Time = 1.0022s | Accuracy = 0.9763
LightGBM (goss): Time = 1.0262s | Accuracy = 0.9771


## 💡 Summary & Core Takeaways

1. **Leaf-Wise Growth**: Focuses compute on splits with highest loss reduction (`Gain`), yielding faster convergence but requiring strict control via `num_leaves` (keep `num_leaves` $\le 2^{\text{max\_depth}}$).
2. **Histogram Binning**: Discretizes continuous variables into $K=255$ bins, reducing memory and split search from $O(\text{data})$ to $O(\text{bins})$.
3. **GOSS**: Accelerates training by keeping high-gradient instances and randomly sampling low-gradient data with reweighting.
4. **EFB**: Merges sparse mutually exclusive features, compressing feature dimensions without loss of information.
5. **Native Categoricals**: Directly splits categorical features in $O(K \log K)$ time without sparse one-hot encoding.